# 5. Treatment Analysis

Survival outcomes by treatment modality and phenotype interaction.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load and prepare data
df = pd.read_csv('LiverMets_Final_Dataset.csv')

complete_tnm = df[
    (df['T_STAGE'].notna()) & (df['T_STAGE'] != 'ND') &
    (df['N_STAGE'].notna()) & (df['N_STAGE'] != 'ND') &
    (df['M_STAGE'].notna()) & (df['M_STAGE'] != 'ND')
]
included = complete_tnm[
    (complete_tnm['SURVIVAL_YEARS'].notna()) & 
    (complete_tnm['SURVIVAL_YEARS'] > 0) &
    (complete_tnm['VITAL_STATUS'].notna())
]

# Define phenotypes
df_tx = included.copy()
df_tx['PHENOTYPE'] = np.nan
mask1 = (df_tx['M_STAGE'] == 'M0') & (df_tx['N_STAGE'].isin(['N0', 'N1']))
df_tx.loc[mask1, 'PHENOTYPE'] = 1
mask2a = (df_tx['M_STAGE'] == 'M0') & (df_tx['N_STAGE'] == 'N2')
mask2b = (df_tx['M_STAGE'] == 'M1') & (df_tx['N_STAGE'].isin(['N0', 'N1']))
df_tx.loc[mask2a | mask2b, 'PHENOTYPE'] = 2
mask3 = (df_tx['M_STAGE'] == 'M1') & (df_tx['N_STAGE'] == 'N2')
df_tx.loc[mask3, 'PHENOTYPE'] = 3

print(f"Cohort: {len(df_tx):,} patients")

## Treatment Distribution

In [ ]:
# Overall treatment distribution
print("Overall Treatment Distribution:")
treatment_counts = df_tx['TREATMENT'].value_counts()
for tx, count in treatment_counts.items():
    pct = 100 * count / len(df_tx)
    print(f"  {tx}: {count:,} ({pct:.1f}%)")

# Treatment by phenotype
print("\n" + "="*70)
print("Treatment Distribution by Phenotype")
print("="*70)

phenotype_names = {1: 'Favourable', 2: 'Intermediate', 3: 'Adverse'}

for ph in [1, 2, 3]:
    cohort = df_tx[df_tx['PHENOTYPE'] == ph]
    print(f"\nPhenotype {int(ph)} ({phenotype_names[ph]}, n={len(cohort):,}):")
    
    for tx in df_tx['TREATMENT'].unique():
        count = (cohort['TREATMENT'] == tx).sum()
        pct = 100 * count / len(cohort) if len(cohort) > 0 else 0
        print(f"  {tx}: {count:,} ({pct:.1f}%)")

## Survival by Treatment (Overall)

In [ ]:
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

# KM curves by treatment
fig, ax = plt.subplots(figsize=(12, 7))

kmf = KaplanMeierFitter()
colors_tx = {'Surgery_Only': '#3498db', 'Surgery+Chemo': '#2ecc71', 
             'Chemo_Only': '#f39c12', 'No_Treatment': '#e74c3c'}

for tx in sorted(df_tx['TREATMENT'].unique()):
    cohort = df_tx[df_tx['TREATMENT'] == tx]
    if len(cohort) > 10:  # Only plot if n > 10
        kmf.fit(cohort['SURVIVAL_YEARS'], cohort['VITAL_STATUS'], 
               label=f'{tx} (n={len(cohort):,})')
        kmf.plot_survival_function(ax=ax, ci_show=True, 
                                   color=colors_tx.get(tx, 'gray'), linewidth=2.5)

ax.set_xlabel('Time (years)', fontsize=12, fontweight='bold')
ax.set_ylabel('Probability of Survival', fontsize=12, fontweight='bold')
ax.set_ylim([0, 1.05])
ax.grid(True, alpha=0.3)
ax.legend(loc='best', fontsize=11)
ax.set_title('Overall Survival by Treatment Modality', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Survival by Treatment and Phenotype (Interaction)

In [ ]:
# KM curves by treatment within each phenotype
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

kmf = KaplanMeierFitter()

for idx, ph in enumerate([1, 2, 3]):
    ax = axes[idx]
    cohort_ph = df_tx[df_tx['PHENOTYPE'] == ph]
    
    for tx in sorted(df_tx['TREATMENT'].unique()):
        cohort_tx = cohort_ph[cohort_ph['TREATMENT'] == tx]
        if len(cohort_tx) > 10:
            kmf.fit(cohort_tx['SURVIVAL_YEARS'], cohort_tx['VITAL_STATUS'],
                   label=f'{tx} (n={len(cohort_tx):,})')
            kmf.plot_survival_function(ax=ax, ci_show=False,
                                       color=colors_tx.get(tx, 'gray'), linewidth=2)
    
    ax.set_xlabel('Time (years)', fontsize=11, fontweight='bold')
    ax.set_ylabel('Probability of Survival', fontsize=11, fontweight='bold')
    ax.set_ylim([0, 1.05])
    ax.grid(True, alpha=0.3)
    ax.legend(loc='best', fontsize=9)
    ax.set_title(f'Phenotype {int(ph)} ({phenotype_names[ph]})', 
                fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## Treatment Effect Summary

In [ ]:
# Summary of survival outcomes
print("\nSurvival Outcomes by Treatment and Phenotype")
print("="*70)

kmf = KaplanMeierFitter()

for ph in [1, 2, 3]:
    print(f"\nPhenotype {int(ph)} ({phenotype_names[ph]}):")
    cohort_ph = df_tx[df_tx['PHENOTYPE'] == ph]
    
    for tx in sorted(df_tx['TREATMENT'].unique()):
        cohort_tx = cohort_ph[cohort_ph['TREATMENT'] == tx]
        
        if len(cohort_tx) > 0:
            kmf.fit(cohort_tx['SURVIVAL_YEARS'], cohort_tx['VITAL_STATUS'])
            deaths = (cohort_tx['VITAL_STATUS'] == 1).sum()
            median_surv = kmf.median_survival_time_
            
            print(f"  {tx}: n={len(cohort_tx):,}, deaths={deaths}, median={median_surv:.1f} yrs")